In [ ]:
import gzip
import io
import os
import pickle

import numpy as np
import pandas as pd
import requests
from ase.visualize import view
from ase.visualize.plot import plot_atoms

from huggingface_hub import hf_hub_download
from pymatgen.io.cif import CifParser
from xtalmet.constants import HF_VERSION
from xtalmet.crystal import Crystal

from monty.serialization import loadfn

# Xtalmet

In [ ]:
# Download generated crystals
samples = {}
for model in ["chemeleon2", "mattergen"]:
	path = hf_hub_download(
		repo_id="masahiro-negishi/xtalmet",
		filename=f"mp20/model/{model}.pkl.gz",
		repo_type="dataset",
		revision=HF_VERSION,
	)
	with gzip.open(path, "rb") as f:
		samples[model] = pickle.load(f)

In [ ]:
# Download training crystals
url = "https://raw.githubusercontent.com/txie-93/cdvae/refs/heads/main/data/mp_20/train.csv"
response = requests.get(url)
train_samples_raw = pd.read_csv(io.StringIO(response.content.decode("utf-8")))
train_samples = []
for _, row in train_samples_raw.iterrows():
	structure = CifParser.from_str(row["cif"]).parse_structures(primitive=True)[0]
	train_samples.append(Crystal.from_Structure(structure))


In [ ]:
RESULTS_DIR = "."  # set your path here

In [ ]:
# Load novelty scores, indices of nearest training samples, and Ehull values
model = "chemeleon2"
with gzip.open(os.path.join(RESULTS_DIR, f"{model}_novelty_scores.pkl.gz"), "rb") as f:
	nov_scores = pickle.load(f)  # (Number of generated samples,)

with gzip.open(
	os.path.join(RESULTS_DIR, f"{model}_nearest_train_indices.pkl.gz"), "rb"
) as f:
	indices_neighbors = pickle.load(f)  # (Number of generated samples,)

with gzip.open(os.path.join(RESULTS_DIR, f"{model}_ehull.pkl.gz"), "rb") as f:
	ehulls = pickle.load(f)  # (Number of generated samples,)

In [ ]:
# Print novel samples and their nearest training samples

# Sort by novelty score (descending)
novel_indices = np.argsort(nov_scores)[::-1]

for idx in novel_indices[:200]:
	crystal = samples[model][
		idx
	]  # xtalmet.crystal.Crystal object (an extension of pymatgen.core.Structure)
	nov_score = nov_scores[idx]
	# exclude compositions having elements "Cs", "Rb", "K"
	if any(
		elem in crystal.composition.as_dict().keys() for elem in ["Cs", "Rb", "K", "Xe"]
	):
		continue
	print(f"Generated sample {idx}:")
	print(
		f"composition: {crystal.composition}, novelty score (AMD): {nov_score:.4f}, Ehull: {ehulls[idx]:.4f} eV/atom"
	)
	neighbor_idx = indices_neighbors[idx]
	neighbor_crystal = train_samples[
		neighbor_idx
	]  # xtalmet.crystal.Crystal object (an extension of pymatgen.core.Structure)
	print(f"Nearest training sample {neighbor_idx}:")
	print(f"composition: {neighbor_crystal.composition}")
	print("-" * 40)

In [ ]:
crystal = samples[model][6444]
view(crystal.to_ase_atoms(), viewer="ngl")

In [ ]:
neighbor_idx = indices_neighbors[idx]
neighbor_crystal = train_samples[neighbor_idx]
view(neighbor_crystal.to_ase_atoms(), viewer="ngl")

# M_LED

In [ ]:
from compute_mled import LocalEnvironmentDiversityCalculator

# Initialize calculator
calc = LocalEnvironmentDiversityCalculator()

In [ ]:
# check @run_mled.py
# # Compute MLED scores for all chemeleon2 samples and save
# from tqdm import tqdm

# model = "chemeleon2"
# gen_structures = samples[model]
# mled_scores = []

# for structure in tqdm(gen_structures, desc="Computing MLED"):
# 	try:
# 		features, ent_pos, ent_poly = calc.featurize_structure(structure)
# 		mled_scores.append(ent_pos + ent_poly)
# 	except Exception as e:
# 		print(f"Failed: {e}")
# 		mled_scores.append(np.nan)

# mled_scores = np.array(mled_scores)

# # Save
# with gzip.open(os.path.join(RESULTS_DIR, f"{model}_mled.pkl.gz"), "wb") as f:
# 	pickle.dump(mled_scores, f)

# print(f"Saved {len(mled_scores)} MLED scores")
# print(f"Mean: {np.nanmean(mled_scores):.4f}, Std: {np.nanstd(mled_scores):.4f}")

In [ ]:
# Load MLED scores
with gzip.open(os.path.join(RESULTS_DIR, f"{model}_mled.pkl.gz"), "rb") as f:
	mled_scores = pickle.load(f)  # numpy array (N,)

print(f"Loaded {len(mled_scores)} MLED scores")
print(f"Mean: {np.nanmean(mled_scores):.4f}, Std: {np.nanstd(mled_scores):.4f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# For MLED vs Novelty
g1 = sns.jointplot(
	x=nov_scores,
	y=mled_scores,
	# kind="reg",
	alpha=0.3,
	# s=10,
	# color="#2E86AB",
)
# g1.plot_joint(sns.kdeplot, color="#E63946", levels=5, linewidths=1.5)
g1.set_axis_labels("Novelty Score (AMD)", "MLED Score", fontsize=12, fontweight="bold")
g1.fig.suptitle("MLED vs Novelty Score", fontsize=13, fontweight="bold", y=1.02)
plt.show()

In [ ]:
ehulls[ehulls > 0.5]

In [ ]:
# tmp
gen_structures = loadfn(
	"../../figures/ablation_rl_structure_div/chemeleon2_rl_dng_struct_div_1_mp_20_1000.json.gz"
)

In [ ]:
# tmp
gen_structures = loadfn("../../benchmarks/dng/chemeleon2_ldm_null_mp_20.json.gz")

In [ ]:
# tmp
gen_structures = samples[model]
for idx, structure in enumerate(gen_structures[:20]):
	features, ent_pos, ent_poly = calc.featurize_structure(structure)
	mled_score = ent_pos + ent_poly
	print(idx, structure.composition, mled_score)

In [ ]:
view(gen_structures[5].to_ase_atoms(), viewer="ngl")

In [ ]:
# mattergen
gen_structures = samples["mattergen"]
for idx, structure in enumerate(gen_structures[:20]):
	features, ent_pos, ent_poly = calc.featurize_structure(structure)
	mled_score = ent_pos + ent_poly
	print(idx, structure.composition, mled_score)

In [ ]:
view(gen_structures[7].to_ase_atoms(), viewer="ngl")